# Twenty questions with an LLM oracle

A hidden animal is picked from about 1,400 options and I can ask an LLM (Qwen2.5-3B) yes/no questions from a fixed pool, with at most 15 queries per animal. I precompute the model's yes/no logit margins for animal and question pairs, keep a soft Bayesian posterior over the animals and always ask the question with the highest information gain. On the dev set it got a mean score of 0.770 (99.3% solved, 11.27 queries on average).

# 🗄️ The Analytical Language of John Wilkins

## Your task

Each hidden creature sits inside a sealed oracle called an **Interactor** — a brass grille over a drawer, holding one animal. You cannot see it. You may put to it one of two kinds of question:

| Call | Returns | What you are asking |
|---|---|---|
| `interactor.ask(question)` | `"yes"` or `"no"` | A yes/no question about the hidden animal. The question must be a line from `questions_pool.txt`. |
| `interactor.guess(animal)` | `"correct"` or `"wrong"` | "Is this the hidden animal?" `"correct"` ends the row. The animal must be a word from `animals_pool.txt`. |

Each question in the pool refers to the creature generically — *"is it a mammal?"*, *"does it live in water?"*, *"can it fly?"* — and the oracle answers about whichever animal is hidden in that drawer.

If you submit a question or animal not in the relevant pool, the oracle refuses without spending its strength: a `ValueError` is raised and your budget is unchanged. Typos cost nothing.

Each drawer will entertain at most **fifteen questions** before the grille falls shut.

### Scoring

For each creature:

```
score = max(0, (1 if you ever guessed correctly else 0) - 0.02 × queries_used)
```

- Correct guess on question 1 → 0.98
- Correct guess on question 5 → 0.90
- Correct guess on question 15 → 0.70
- Never correct → 0

Your score is the mean across all creatures in a test set. Tune on `dev`, then run the final cell to get your **`test1`** score — that summary table is what you submit a screenshot of. The organizers keep a second, **hidden** test set for official grading, so a solution that genuinely deduces (rather than overfits `dev`/`test1`) is what scores well.

## The oracle

In plain language, the oracle inside each Interactor is a local language model — by default, `Qwen/Qwen2.5-3B-Instruct`. When you call `ask(question)` it prompts the model with:

```
You are answering a question about one specific animal.
The animal is: <hidden animal>.
Answer with a single word, yes or no.
Question: <question>
```

at temperature 0, parses the first word of the reply, and returns `"yes"` or `"no"` to your code.

The model is deterministic (the same `(animal, question)` pair always gives the same answer) and runs entirely inside the Interactor. You are free to run the same model in your own code to **predict** what it will say without spending the oracle's strength — that is a large part of what makes a clever solution. Note the oracle answers from the model's *beliefs* about the animal, which are usually right but not infallible; a good solution is robust to the occasional surprising answer.

## Step 1: Setup

The dataset and helper code (`interactor.py`, `evaluate.py`, the two pools, the dev/test CSVs) live in the shared **`IOAI-2026/AnimalDeduction/dataset`** Drive folder. The cell below just downloads them into Colab — no sign-in, no shortcuts, just run it. Use a **GPU** runtime: *Runtime → Change runtime type → T4* (free tier is enough).

In [31]:
from google.colab import drive
drive.mount('/content/drive')
import numpy as np
import pandas as pd

Mounted at /content/drive


In [32]:
import os
import shutil
import zipfile
import sys

source_path = '/content/drive/MyDrive/dataset_HT3_2026.zip'
target_dir = '/content/animaldeduction'

# 1. Create target directory
os.makedirs(target_dir, exist_ok=True)

# 2. Extract contents directly into the target folder
print("Unzipping dataset...")
with zipfile.ZipFile(source_path, 'r') as zip_ref:
    zip_ref.extractall(target_dir)

print(f"Successfully extracted dataset contents to {target_dir}!")

nested_dir = '/content/animaldeduction/dataset'
target_dir = '/content/animaldeduction'

if os.path.exists(nested_dir):
    # Move all files/folders from /content/animaldeduction/dataset -> /content/animaldeduction
    for item in os.listdir(nested_dir):
        shutil.move(os.path.join(nested_dir, item), os.path.join(target_dir, item))

    # Remove the now-empty nested dataset folder
    os.rmdir(nested_dir)
    print("Successfully flattened the dataset directory structure!\n")

# Verify what files are now in /content/animaldeduction
print("Files now in /content/animaldeduction:")
for file in os.listdir(target_dir):
    print(f" - {file}")

# Set target folder
LOCAL_DIR = '/content/animaldeduction'

# 1. Tell Python to look in this folder when importing modules like interactor or evaluate
if LOCAL_DIR not in sys.path:
    sys.path.insert(0, LOCAL_DIR)

# 2. Change the working directory so relative file paths work automatically
os.chdir(LOCAL_DIR)

print('Current working directory:', os.getcwd())
print('Available files:', sorted(os.listdir()))

Unzipping dataset...
Successfully extracted dataset contents to /content/animaldeduction!
Successfully flattened the dataset directory structure!

Files now in /content/animaldeduction:
 - test1.csv
 - animals_pool.txt
 - dev.csv
 - evaluate.py
 - interactor.py
 - questions_pool.txt
Current working directory: /content/animaldeduction
Available files: ['animals_pool.txt', 'dev.csv', 'evaluate.py', 'interactor.py', 'questions_pool.txt', 'test1.csv']


## Step 2: Load data and try the oracle

The `Interactor` owns the hidden gold animal and runs a local LLM (Qwen 2.5 3B Instruct by default) to answer yes/no questions about it. The first `Interactor(...)` instantiation triggers the LLM download (~6 GB on first run, takes 30-60 s on T4). Every subsequent Interactor reuses the same LLM that's already loaded in memory.

In [34]:
import random
import numpy as np
import pandas as pd
import torch

from interactor import Interactor
from evaluate import evaluate, load_pools

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

animals_pool, questions_pool = load_pools()
print(f'animals_pool size:   {len(animals_pool):>6}  (e.g. {animals_pool[:5]})')
print(f'questions_pool size: {len(questions_pool):>6}  (e.g. {questions_pool[:3]})')

# Sanity probe: create one Interactor and ask two questions about an octopus.
probe = Interactor(gold_animal='octopus', animals_pool=animals_pool, questions_pool=questions_pool)
print("\nask('is it a mammal?')      ->", probe.ask('is it a mammal?'))
print("ask('does it live in water?') ->", probe.ask('does it live in water?'))
print('Queries used:', probe.queries_used, '/', probe.budget)

# The model loads in bfloat16, which a T4 (Turing) cannot accelerate. Convert to
# float16 (T4 has fp16 tensor cores) -> identical answers, several times faster.
import torch
if torch.cuda.is_available() and next(Interactor._model.parameters()).dtype != torch.float16:
    Interactor._model = Interactor._model.half()
    print('oracle model -> float16 (faster on T4)')

Device: cuda
animals_pool size:     1472  (e.g. ['lion', 'tiger', 'leopard', 'snow leopard', 'cheetah'])
questions_pool size:    559  (e.g. ['is it a mammal?', 'is it a bird?', 'is it a reptile?'])
  [interactor] loading Qwen/Qwen2.5-3B-Instruct on cuda...


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

... (3 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

  [interactor] LLM ready.

ask('is it a mammal?')      -> no
ask('does it live in water?') -> yes
Queries used: 2 / 15


## Step 3: Solution interface

Your solution is a class with two methods:

- `__init__(self, animals_pool, questions_pool)` — runs once. Load models, precompute tables, etc.
- `solve(self, interactor)` — runs once per test row. Use the oracle to identify the hidden animal.

Inside `solve`, you have:

```
interactor.ask(question)      -> 'yes' or 'no'        (question must be in questions_pool)
interactor.guess(animal)      -> 'correct' or 'wrong' (animal must be in animals_pool)
interactor.is_done()          -> True after a correct guess or budget exhausted
interactor.remaining_budget() -> int
```

**Scoring per row**: `score = max(0, (1 if you ever guess correctly else 0) - 0.02 * total_queries)`.

Budget is **15 questions** per row. Information theory: `log₂(1400) ≈ 10.5 bits`, each yes/no answer is at most 1 bit — so ~11 well-chosen questions plus 1 final guess fit the budget, *if* each question splits the remaining candidates in half. Most don't: *"does it have a backbone?"* sounds decisive but the model calls a great many creatures vertebrates. A good question splits the *remaining* candidates roughly in half, given everything you have already learned — so the right next question depends on the answers so far.

### Baseline: random guessing (the floor)

Ignores `ask()` entirely. Just guesses random animals until the budget runs out. Expected score: ~0 (15 random guesses out of ~1,400 candidates ≈ 1% solve rate). Any reasonable solution needs to beat this by a lot.

In [35]:
import inspect
import numpy as np
import torch
from transformers import DynamicCache

JUDGE_PROMPT = (
        "You are answering a question about one specific animal.\n"
        "The animal is: {animal}.\n"
        "Answer with a single word, yes or no.\n"
        "Question: {question}"
    )

K_Questions = 24
K_Animals = 24

In [38]:
import inspect
from tqdm import tqdm
from transformers import DynamicCache
from IPython.display import clear_output

K_QUESTIONS = 24     # how many table columns.  32 if you have GPU headroom.
N_ANCHOR    = 24     # animals sampled to estimate yes-rates
TAU         = 0.12   # confidence -> probability scale
FLOOR       = 5e-5   # a disagreement costs -log(5e-5) = 9.9, never infinity
ASK_MARG = 3.0


class MySolution:
    def __init__(self, animals_pool, questions_pool, seed=0):

        Interactor._ensure_llm() # sets tokenizer, modedl and so on but does nothing if set already

        self.animals = [a.strip().lower() for a in animals_pool] # .strip() removes leading and trailing characs
        self.questions = [q.strip().lower() for q in questions_pool]
        self.tokenizer, self.model = Interactor._tokenizer, Interactor._model
        self.yes_ids = self._ids(["Yes", "yes", "YES"])
        self.no_ids  = self._ids(["No",  "no",  "NO"])
        params = inspect.signature(self.model.forward).parameters # This returns all the args that the call has -- and specifically
        # Checks if logits_to_keep : int | torch.Tensor = 0 exsists, (which on here does)
        self.logits_to_keep = ({'logits_to_keep' : 1} if "logits_to_keep" in params else
                               {'num_logits_to_keep' : 1} if "num_logits_to_keep" in params else {})

        self.pad = self.tokenizer.pad_token_id
        self.A_len, self.Q_len = len(self.animals), len(self.questions)

        self._prep_suf_and_pref()

                # === STEP 1: which questions are worth asking? ===
        # Try all 559 questions on a SPREAD-OUT sample of 24 animals and see how
        # often each says yes.  Spread out matters: the pool is sorted by category,
        # so animals[:24] is 24 cats and "is it a feline?" would look amazing.


        n = len(self.animals)
        animal_samples = [i * n // N_ANCHOR for i in range(N_ANCHOR)]
        yes_rate = self._table(animal_samples, [i for i in range(len(self.questions))])[0].mean(0) # if i have b[M, N] will get b[N] by mean
        # do neg as -1.0, -0.5, .... will be best in sort
        # 3. Calculate Shannon entropy for each question
        entropy = self._entropy(yes_rate)  # shape [Q] -- which is like 559

        # 4. Pick indices of the top K questions with highest entropy
        top_K_questions = np.argsort(-entropy)[:K_QUESTIONS].tolist()

        self.top_k_q = top_K_questions

        # === STEP 2: the real table, every animal x those questions ===
        self.bits, self.marg = self._table(range(n), self.top_k_q)

        self.improve_oracle_speed()

        # === STEP 3: turn confidence into probability ===
        # marg is the model's confidence.  f = "chance this cell is wrong".  It shrinks
        # as confidence grows but NEVER reaches 0 -- that floor is what stops one
        # surprising answer from deleting the true animal forever.

        f = np.clip(0.5 * np.exp(-np.abs(self.marg) / TAU), FLOOR, 0.5) # P(chance of being wrong)
        # 0.5 * sets the value at zero margin. Correct intent - f(0) = 0.5 = coin flip.
        # TAU = 0.12. set by the fp16 noise scale, validated against measured flip rates.

        self.p_yes   = np.where(self.bits, 1 - f, f)  #(self.bits, ifTrue, ifFalse)
        # if self.bits is True, so chance its wrong is f, so P(answers yes) is 1-f
        # if self.bits is False, so chance its wrong is f, so P(answers yes) is f
        self.log_yes = np.log(self.p_yes)
        self.log_no  = np.log(1 - self.p_yes)

     # --- helpers ----
    def _ids(self, strs):
        out = []
        for s in strs:
          i = self.tokenizer.encode(s, add_special_tokens=False)
          if len(i) == 1:
            out.append(i[0])
        return sorted(set(out))

    def _entropy(self, p):
      p = np.clip(p, 1e-9, 1 - 1e-9)
      return - (p * np.log2(p) + (1-p) * np.log2(1-p))


    def _kv_out(self, cache):                       # pull raw (keys, values) out of a cache
      if hasattr(cache, "to_legacy_cache"):
          return list(cache.to_legacy_cache())
      return [(l.keys, l.values) for l in cache.layers]


    def _kv_in(self, kv):
        kv = tuple(kv)                           # wrap them in a NEW cache object
        if hasattr(DynamicCache, "from_legacy_cache"):
            return DynamicCache.from_legacy_cache(kv)
        c = DynamicCache()
        for i, (k, v) in enumerate(kv):
            c.update(k, v, i)
        return c

    def _pos(self, mask):
        """Real position of each token, skipping left padding."""
        return (mask.cumsum(dim=-1) - 1).clamp(min=0) # cummultative sum
    """ for att mask
       [[0, 0, 1, 1],
        [1, 1, 1, 1],
        [1, 1, 1, 1]]
        we have pos:
        [[0, 0, 0, 1],
        [0, 1, 2, 3],
        [0, 1, 2, 3]]
    """

    def _pad(self, seqs):
        # we do LEFT padding
        L_max = 0
        for i in range(len(seqs)):
          L_max = max(L_max, len(seqs[i]))
        ids = np.full((len(seqs), L_max), self.pad, np.int64)
        masks = np.zeros((len(seqs), L_max), np.int64)
        for i, t in enumerate(seqs):
          ids[i, L_max - len(t):] = t
          masks[i, L_max - len(t):] = 1
        return torch.as_tensor(ids).to('cuda'), torch.as_tensor(masks).to('cuda') # zero copy and faster than tensor.tensor()


    # -- generate suf and pref for KV-Cache --

    # -- what we do next is to use KV cache we neeed to seperate the chat prompt info pref (before) and suffixes (after) so we --
    # -- seperate it into    <|im_start|> system.... or no.\nQuestion:    and    is it a reptile??<|im_end|>\n<|im_start|>assistant\n


    def _chat(self, animal, question): # you can do dir(Interactor._tokenizer) to see it urself
      return self.tokenizer.apply_chat_template(
          [{'role': 'user', 'content': JUDGE_PROMPT.format(animal = animal, question = question)}],
          # Here because in JUDGE_PROMPT I put {animal} and {question}, I can change their content and values I put inside string
          tokenize = False, add_generation_prompt=True)
          # And the 'role" : 'user' is typical, in llms they often do [{'role':'system', 'content' : ...}, {'role':'user', ...}]
          # add_generation_prompt appends <|im_start|>assistant\n telling model to generate its response the following token.

    def _prep_suf_and_pref(self):
      reference_anim = self.animals[0] # we do add_special_tokens=False as apply_chat_template adds them already so no error or dupl
      toks = [self.tokenizer(self._chat(reference_anim, q), add_special_tokens=False).input_ids # deleted add_special_tokens = False
                           for q in self.questions]
      # now that we generated all ques with same anim, we find longest common pref
      longest_pref = len(toks[0])
      for t in toks[1:]:
        i = 0
        while i < min(longest_pref, len(t), len(toks[0])) and t[i] == toks[0][i]:
          i += 1
        #print(i)
        longest_pref = min(longest_pref, i)
      #print(longest_pref)
      self.tails = [tt[longest_pref:] for tt in toks]
      suf_len_first = len(self.tails[0])

      self.heads = [self.tokenizer(self._chat(a, self.questions[0]),
                        add_special_tokens=False).input_ids[:-suf_len_first] for a in self.animals]
                        # [:-x] trims the last x and leaves everything else

    @torch.no_grad()
    def _table(self, animal_idx, columns, batch_sz = 64):
      # so what we do is firstly is get ids on cuda
      yes_t = torch.tensor(self.yes_ids, device='cuda')
      no_t = torch.tensor(self.no_ids, device='cuda')
      R, K = len(animal_idx), len(columns)

      # initiate true/false and by what margain

      bits = np.zeros((R, K), bool)
      margains = np.zeros((R, K), np.float32)
      # so now what we do is sort animal prefixes lens, and given same prefs collect them in a batch of size +- 128 until
      # not left and just iterate over all questions -- we KV-cache the animal batche and just rerun for every question
      tails = [torch.tensor(self.tails[i], device='cuda') for i in columns]

        # this is iterating through animals, padding them to the left and using cache
        # rows = animals, columns = questions_idx
      for s in tqdm(range(0, R, batch_sz)):
        rows = animal_idx[s : s + batch_sz]
        head, mask = self._pad([self.heads[a] for a in rows])
        G = len(rows)
        cache = DynamicCache()
        # --- run the animal half once, keep the cache ---
        self.model(input_ids = head, attention_mask=mask, position_ids = self._pos(mask),
                   past_key_values = cache, use_cache = True, **self.logits_to_keep)
        kv = self._kv_out(cache)

        # --- run each question tail against that cache ---
        for c, tail in enumerate(tails):
          Tail_len = tail.shape[0]
          mask2 = torch.cat([mask, torch.ones((G, Tail_len), dtype=torch.long, device='cuda')], dim=1) # along the lenghts of the seq
          outs = self.model(input_ids = tail.expand(G, Tail_len), # same tail for all animals
                                                  #-- input_ids shape dictates position_ids shape (should be same)
                           attention_mask = mask2,       # covers cache + new tokens
                           position_ids = self._pos(mask2)[:, -Tail_len:], # KV values stored pos ids already so only do for new toks
                           past_key_values = self._kv_in(kv),
                           use_cache=False, # whether the model returns the Key-Value (KV) activations for the current forward pass
                           **self.logits_to_keep) # now of shape [B, 1, 153232]

          # margain -- best yes logit - best no logit
          #.float() BEFORE subtracting: fp16 is too coarse for the small margins.
          ye = outs.logits[:, -1, self.yes_ids].float().max(-1).values # along axis -1
          no = outs.logits[:, -1, self.no_ids].float().max(-1).values # along axis -1
          d = (ye - no).cpu().numpy()
          bits[s:s+G, c], margains[s:s+G, c] = d > 0, d # we do s:s+G as we are not guaranteed that s+batch_size does not exceed
      return bits, margains

    def improve_oracle_speed(self): # inner is just the same as self. of the outer function -- its an instance
      if getattr(Interactor, '_patched', False):  # -> Returns False (not patched yet) reverted to default False, before not assigned
        return      ## -- to False as we have written here
      model, tokenizer, chat = self.model, self.tokenizer, self._chat
      l_to_keep = self.logits_to_keep
      YES, NO, orig = set(self.yes_ids), set(self.no_ids), Interactor._llm_yes_no
      # because inside orig there is different 'self' instance -- we name it inner

      def fast(inner, question):
        inps = torch.tensor(tokenizer([chat(inner.gold, question)],
                add_special_tokens=False).input_ids, # or just `input_ids` list directly
                device='cuda')
        with torch.no_grad():
          top_id = int(model(input_ids = inps, **l_to_keep).logits[:, -1, :].argmax(-1))
        return True if top_id in YES else False if top_id in NO else orig(inner, question)
        #You replaced Interactor._llm_yes_no with your new fast function!
        #If the model predicts a token that isn't a simple "Yes" or "No" (a fallback case),
        # it fallbacks to orig(inner, question) -- the same args as of _llm_yes_no before

      Interactor._llm_yes_no = fast
      Interactor._patched = True


    def solve(self, interactor):
        scores = np.zeros(len(self.animals)) # log probs
        have_asked = np.zeros(len(self.top_k_q), bool)

        while interactor.remaining_budget() > 0 and interactor.is_done() == False:
          w = np.exp(scores - scores.max()) #  np.exp undoes the log so now normal probabilities
          p = w / w.sum()

          entropy_score = (-(p * np.log2(np.maximum(p, 1e-300))).sum()) # so what this is doing is taking the entropy score
          live_left = float(2 ** entropy_score) # and doing 2 ** that so we can like sucpect how mcuh of the animals +- cnacdidates are left
          """
          Start from the fact you already know: to pick one of N equally likely animals you need log₂(N) yes/no answers. With 1472 animals that's 10.52 bits.

          Entropy runs that forward: it tells you how many bits you still need. Perplexity runs it backward:

          uniform over N   →   H = log₂(N)   →   2^H = 2^(log₂ N) = N

          The 2^ and the log₂ cancel, so you get N back exactly. That's the whole trick.
          """
          ask = False
          # .all() -- returns True if all elements in a iterable are truthy
          if interactor.remaining_budget() > 1 and not have_asked.all() and live_left > ASK_MARG:
            # For each question: how likely is "yes", given what I believe now?
                # Closest to 0.5 = the answer I can least predict = the most information.
                p_of_yes = p @ self.p_yes # so normal p probs @ self.p_yes -- we want to see how good the sum splits up
                """
                (m, n) @ (n, k)  →  (m, k)        the n vanishes

                Here the left side is 1-D, so m is absent:
                (1472,) @ (1472, 24)  →  (24,)
                ↑          ↑
                n=1472    n=1472    match → contracted
                                ↑
                                k=24  survives
                """
                g = np.abs(p_of_yes - 0.5)
                g[have_asked] = np.inf
                j = int(np.argmin(g))
                ask = min(p_of_yes[j], 1-p_of_yes[j]) > 1e-3 # A check on the winner. If even the best remaining question
                #is 99.9% predetermined, it carries ~0 bits, so guessing is the better use of that query.

          if ask == True:
            said_yes = interactor.ask(self.questions[self.top_k_q[j]]) == "yes"
            have_asked[j] = True
            # Bayes: multiply by P(this answer | animal).  In logs, that's +=.
            scores += self.log_yes[:, j] if said_yes else self.log_no[:, j]
          else:
            best = int(np.argmax(p))
            interactor.guess(self.animals[best])
            scores[best] = -1e18

# ---- cell B: build once (~4 min) ----
model = MySolution(animals_pool, questions_pool)

100%|██████████| 23/23 [03:22<00:00,  8.81s/it]


In [39]:
# ---- cell C: iterate, instant ----
model.__class__ = MySolution        # pick up any edits from cell A
evaluate(model, 'dev.csv')

  25/150 rows  mean_score=0.7792  (16.2s)
  50/150 rows  mean_score=0.7768  (32.6s)
  75/150 rows  mean_score=0.7760  (48.7s)
  100/150 rows  mean_score=0.7682  (64.9s)
  125/150 rows  mean_score=0.7688  (80.9s)
  150/150 rows  mean_score=0.7700  (96.3s)

  Dataset:       dev.csv
  Mean score:    0.7700
  Solved rate:   99.3%
  Mean queries:  11.27 / 15
  Wall time:     96.3s


{'n': 150,
 'mean_score': 0.77,
 'solved_rate': 0.9933333333333333,
 'mean_queries': 11.266666666666667,
 'per_row':                       gold  queries_used  solved  score
 0                 aardwolf            11       1   0.78
 1                  abalone            12       1   0.76
 2             african lion            11       1   0.78
 3                   alpaca            11       1   0.78
 4      american black bear            11       1   0.78
 ..                     ...           ...     ...    ...
 145             weaver ant            12       1   0.76
 146          whip scorpion            10       1   0.80
 147              wood frog            12       1   0.76
 148  yellow-bellied marmot            12       1   0.76
 149              zebrafish            12       1   0.76
 
 [150 rows x 4 columns]}

In [45]:
for tau in (0.06, 0.12, 0.25):
    for ask in (2.5, 3.0, 4.0, 5.0):
        f = np.clip(0.5*np.exp(-np.abs(model.marg)/tau), FLOOR, 0.5)
        model.p_yes = np.where(model.bits, 1-f, f)
        model.log_yes, model.log_no = np.log(model.p_yes), np.log(1-model.p_yes)
        globals()['ASK_MARG'] = ask
        print(tau, ask, evaluate(model, 'dev.csv')['mean_score'])

TypeError: Interactor.ask() missing 1 required positional argument: 'question'

### Reference: a non-adaptive 20-questions sketch

This reference shows the *shape* of a real solution without giving away the points. In `__init__` it precomputes, with its own copy of the model, the oracle's yes/no answer to a small **fixed** list of broad questions for every animal — a bit-vector per animal. In `solve` it asks those same fixed questions, reads off the oracle's bit-vector, and guesses the animals whose precomputed vector is closest.

It works, but it's deliberately weak: the questions are the **same for every row** (not chosen adaptively to split the *remaining* candidates), and it uses only a handful. Beating it is mostly about (1) precomputing the full `animal × question` table and (2) choosing each next question *greedily* to most evenly split the animals still consistent with the answers so far. That's your job in Step 4.

> Precomputing even this small table calls the model a few thousand times (~5-15 min on T4). Skip this cell if you just want to get to your own solution — it is only a reference.

### 💡 Speed tip

Building the animal×question table by calling `interactor.ask()` **one at a time** is slow.
You can make your precompute **much** faster — without changing any answers — by **batching**
your model calls (many prompts through the model per forward pass). That optimization is up to you.

## Step 4: Your solution

Replace the body of `MySolution.solve` (and `__init__` if you precompute anything) with your strategy. Iterate on `dev.csv` until you're happy with the score, then jump to Step 5 to evaluate on test1 + test2.

**The intended approach:**
1. In `__init__`, precompute once — with your own copy of the model — the oracle's yes/no answer for every `(animal, question)` pair you care about. This costs no oracle budget.
2. In `solve`, keep a set of candidate animals still consistent with the answers so far. At each step pick the **question whose answer most evenly splits that set** (maximize information gain), ask it, and shrink the set. Guess when one candidate dominates or the budget is nearly gone.
3. Be robust: the oracle occasionally answers in a way your table didn't predict. Don't let one surprising bit eliminate the true animal forever.

## Step 5: Final scoring

Once you're happy with your dev score, run this cell. It scores `dev` and `test1` (and `test2` automatically, if that file is present). The **FINAL** line — the *n*-weighted mean over the available test split(s) — is what you submit a screenshot of.

> The organizers also score your submitted `MySolution` on a separate **hidden** test set that is not included here. Aim for a strategy that deduces the animal from scratch each row, so it transfers to unseen creatures.